<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/2_2_Metric_Methods_k%E2%80%91Nearest_Neighbors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part II. Classification. Metric Methods: k‑Nearest Neighbors

# Глава 5. Метрические методы: k‑ближайших соседей

В предыдущей главе мы изучили логистическую регрессию — параметрический метод классификации, который предполагает, что логарифм шансов линейно зависит от признаков. Это мощный и интерпретируемый инструмент, но его ограничение очевидно: реальная граница между классами редко бывает строго линейной. Мы могли бы добавить полиномиальные признаки или базисные функции, но как тогда выбрать их вид? И что, если мы вообще не хотим делать никаких предположений о форме границы?
>
> **k‑ближайших соседей (k‑NN)** предлагает радикально иной подход: никаких параметров, никакой явной функции, никакого обучения в привычном смысле. Вместо этого алгоритм запоминает все обучающие данные и классифицирует новый объект, глядя на его «соседей». Это непараметрический метод, который служит естественным мостом между линейными моделями и полностью гибкими нелинейными подходами.
| Характеристика | Логистическая регрессия | k‑ближайших соседей |
|---------------|-------------------------|---------------------|
| Тип | Параметрическая | Непараметрическая |
| Обучение | Оптимизация весов (градиент) | Запоминание данных |
| Предсказание | Вычисление сигмоиды | Поиск соседей |
| Интерпретация | Коэффициенты → шансы | Нет явной интерпретации |
| Граница | Гладкая (линейная/полиномиальная) | Кусочно-постоянная |
| Чувствительность к выбросам | Низкая (регуляризация) | Высокая (особенно при малых k) |
| Проклятие размерности | Умеренное | Катастрофическое |

 В этой главе мы погрузимся в геометрию классификации, изучим компромисс смещения и дисперсии на примере k‑NN и увидим, как простой принцип «большинство соседей» приводит к удивительно глубоким результатам.






## 1. Идея метода: геометрия и формализация

Фундаментальный принцип метода ближайших соседей можно выразить максимой: **«скажи мне, кто твои соседи, и я скажу, кто ты»**. В задаче классификации новый объект получает метку, которую имеет большинство из $k$ его ближайших соседей в обучающей выборке; для регрессии откликом служит среднее (или взвешенное среднее) значений целевой переменной этих соседей. Интуитивное обоснование очевидно: объекты, расположенные в признаковом пространстве достаточно близко друг к другу, с высокой вероятностью обладают схожими свойствами.

Однако за внешней простотой скрывается строгая вероятностная логика. Пусть среди $k$ ближайших соседей точки $x$ ровно $k_c$ объектов принадлежат классу $c$. Тогда решающее правило большинства можно представить в эквивалентном виде

$$
\hat{y}(x) = \arg\max_{c} \frac{k_c}{k}.
$$

Дробь $k_c / k$ есть не что иное, как **непараметрическая оценка апостериорной вероятности**:

$$
\hat{P}(Y = c \mid X = x) = \frac{k_c}{k}. \tag{1.1}
$$

Таким образом, метод k‑NN осуществляет локальную оценку вероятностей классов и принимает решение в пользу наиболее вероятного из них. Это напрямую связывает его с байесовским решающим правилом: если оценка (1.1) состоятельна, то при неограниченном росте объёма данных классификатор k‑NN сходится к оптимальному байесовскому классификатору (подробнее см. раздел 3).

### 1.1 Формальное определение

Пусть обучающая выборка состоит из $n$ пар $\{(x_i, y_i)\}_{i=1}^n$, где $x_i \in \mathbb{R}^D$ — вектор признаков, а $y_i$ принимает значения из конечного множества $\{1, 2, \dots, C\}$ (классификация) или $y_i \in \mathbb{R}$ (регрессия). Для нового объекта $x$ определим множество индексов его $k$ ближайших соседей относительно наперёд заданной метрики $d$:

$$
N_k(x) = \{ i_{(1)}, i_{(2)}, \dots, i_{(k)} \} \subset \{1,\dots,n\},
$$

где индексы упорядочены по возрастанию расстояния:

$$
d(x, x_{i_{(1)}}) \le d(x, x_{i_{(2)}}) \le \dots \le d(x, x_{i_{(k)}}) \le d(x, x_j), \quad \forall j \notin N_k(x). \tag{1.2}
$$

**Классификация** выполняется по мажоритарному принципу:

$$
\hat{y}(x) = \arg\max_{c \in \{1,\dots,C\}} \sum_{i \in N_k(x)} \mathbb{1}(y_i = c), \tag{1.3}
$$

где $\mathbb{1}(\cdot)$ — индикаторная функция, равная 1, если условие истинно, и 0 в противном случае. При чётном $k$ возможна ситуация равенства голосов; стандартные рекомендации предписывают в таких случаях либо уменьшать $k$ на единицу, либо использовать случайный выбор среди классов, набравших одинаковое число голосов. На практике для бинарной классификации часто выбирают нечётные значения $k$.

**Регрессия** использует простое арифметическое среднее:

$$
\hat{y}(x) = \frac{1}{k} \sum_{i \in N_k(x)} y_i. \tag{1.4}
$$

Это локально-постоянная аппроксимация функции регрессии $m(x) = \mathbb{E}[Y \mid X = x]$. Действительно, вводя веса

$$
W_i(x) = \begin{cases}
1/k, & i \in N_k(x), \\
0, & \text{иначе},
\end{cases}
$$

мы получаем представление

$$
\hat{m}(x) = \sum_{i=1}^n W_i(x) \, y_i,
$$

которое является частным случаем **ядерной оценки Надарая–Ватсона** (Nadaraya, 1964; Watson, 1964) с равномерным ядром и адаптивной шириной окна, определяемой локальной плотностью данных через число соседей $k$. В отличие от методов с фиксированной шириной окна, здесь окно автоматически расширяется в областях малой плотности и сужается в плотных областях, что обеспечивает определённую устойчивость оценок.

### 1.2 Метрика и её роль

Центральным понятием метода является **расстояние** между объектами. Функция $d: \mathbb{R}^D \times \mathbb{R}^D \to \mathbb{R}_+$ называется метрикой, если для любых $x, x', x''$ выполняются аксиомы:

1. **Невырожденность**: $d(x, x') \ge 0$, причём $d(x, x') = 0 \iff x = x'$;
2. **Симметричность**: $d(x, x') = d(x', x)$;
3. **Неравенство треугольника**: $d(x, x') \le d(x, x'') + d(x'', x')$.

Соблюдение этих свойств гарантирует, что геометрическая интуиция о «близости» не противоречит формальным операциям. В практических реализациях иногда используются и меры, не удовлетворяющие всем трём аксиомам (например, косинусное расстояние), что требует осторожности при построении индексов и интерпретации окрестностей.

Наиболее общим семейством метрик в $\mathbb{R}^D$ является **расстояние Минковского** порядка $p \ge 1$:

$$
\|x - x'\|_p = \left( \sum_{j=1}^D |x^{(j)} - x'^{(j)}|^p \right)^{1/p}. \tag{1.5}
$$

Три важнейших частных случая:

- $p=1$ — **манхэттенское расстояние** (сумма модулей разностей);
- $p=2$ — **евклидово расстояние** (геометрическая длина);
- $p \to \infty$ — **равномерная норма** $\|x - x'\|_\infty = \max_j |x^{(j)} - x'^{(j)}|$.

В двумерном пространстве наглядно видна трансформация единичного шара $\{u : \|u\|_p \le 1\}$ при изменении $p$: при $p=1$ это ромб с вершинами на осях координат, при $p=2$ — круг, при $p=4$ — фигура, близкая к квадрату со скруглёнными углами, а в пределе $p \to \infty$ — квадрат со стороной $2$, параллельной координатным осям. Чем меньше $p$, тем сильнее метрика «поощряет» большие разности по отдельным координатам и может игнорировать малые расхождения по остальным; большие $p$, напротив, стремятся уравнять вклад всех измерений.

Для **текстовых и высокоразмерных разреженных данных** часто применяют **косинусное расстояние**:

$$
d_{\cos}(x, x') = 1 - \frac{\langle x, x' \rangle}{\|x\|_2 \|x'\|_2}, \tag{1.6}
$$

измеряющее угол между векторами. Оно инвариантно к масштабу признаков и хорошо зарекомендовало себя в моделях представления текстов (bag‑of‑words, TF‑IDF). Однако косинусная мера не удовлетворяет неравенству треугольника без дополнительной нормировки, и её использование в некоторых структурах данных (например, в KD-деревьях) требует модификаций.

**Практическая рекомендация.** В пространствах размерности $D > 10$ разница между евклидовой и манхэттенской метриками зачастую нивелируется вследствие концентрации меры (см. раздел 3). Тем не менее выбор метрики должен быть продиктован природой данных: для однородных числовых признаков, измеренных в одинаковых единицах, предпочтительна евклидова метрика; для гетерогенных величин — манхэттенское расстояние либо взвешенные варианты, в которых веса подбираются по важности признаков.

### 1.3 Подробные примеры в двумерном случае

Чтобы прочувствовать механизм k‑NN во всей полноте, разберём работу алгоритма «вручную» на миниатюрном наборе данных в $\mathbb{R}^2$ отдельно для классификации и для регрессии.

#### 1.3.1 Классификация

**Обучающая выборка** из четырёх точек:

| Точка | $x_1$ | $x_2$ | Класс |
|-------|-------|-------|-------|
| A     | 1     | 2     | 0     |
| B     | 2     | 3     | 0     |
| C     | 5     | 5     | 1     |
| D     | 6     | 4     | 1     |

Пусть новый объект $x = (3, 3)$, и мы выбрали $k = 3$. Вычислим евклидовы расстояния $d_2(x, \cdot)$:

$$
\begin{aligned}
d_2(x, A) &= \sqrt{(3-1)^2 + (3-2)^2} = \sqrt{4 + 1} = \sqrt{5} \approx 2.236, \\
d_2(x, B) &= \sqrt{(3-2)^2 + (3-3)^2} = \sqrt{1 + 0} = 1, \\
d_2(x, C) &= \sqrt{(3-5)^2 + (3-5)^2} = \sqrt{4 + 4} = \sqrt{8} \approx 2.828, \\
d_2(x, D) &= \sqrt{(3-6)^2 + (3-4)^2} = \sqrt{9 + 1} = \sqrt{10} \approx 3.162.
\end{aligned}
$$

Упорядочиваем точки по возрастанию расстояния: $B$ (1.000), $A$ (2.236), $C$ (2.828), $D$ (3.162).  
Три ближайших соседа — $B, A, C$. Среди них два объекта класса 0 ($A$ и $B$) и один объект класса 1 ($C$). Оценки апостериорных вероятностей:

$$
\hat{P}(Y=0 \mid x) = \frac{2}{3} \approx 0.667,\qquad
\hat{P}(Y=1 \mid x) = \frac{1}{3} \approx 0.333.
$$

Максимальную вероятность имеет класс 0, поэтому $\hat{y}(x) = 0$.  
При $k=2$ соседями были бы $B$ и $A$ (оба класса 0) — результат не изменился бы. При $k=4$ голоса делятся поровну (2:2), и необходимо дополнительное правило разрешения ничьей.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Обучающая выборка (точки A, B, C, D) и классы
X_train = np.array([
    [1, 2],   # A
    [2, 3],   # B
    [5, 5],   # C
    [6, 4]    # D
])
y_train = np.array([0, 0, 1, 1])   # 0 и 1

# Новая точка
x_new = np.array([3, 3])

# Параметр k
k = 3

# Евклидовы расстояния и поиск k ближайших соседей
distances = np.linalg.norm(X_train - x_new, axis=1)
idx_sorted = np.argsort(distances)
neighbors_idx = idx_sorted[:k]

# Вывод в консоль
print("Расстояния до точек A,B,C,D:", distances)
print("Индексы k ближайших соседей:   ", neighbors_idx)
print("Классы выбранных соседей:       ", y_train[neighbors_idx])
pred_class = np.bincount(y_train[neighbors_idx]).argmax()
print("Предсказанный класс (большинство):", pred_class)

# Визуализация
fig, ax = plt.subplots(figsize=(7, 5))

# Все обучающие точки
ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
           cmap='bwr', edgecolors='k', s=120, zorder=5,
           label='Обучающие точки (класс)')
# Новая точка
ax.scatter(x_new[0], x_new[1], c='lime', marker='*',
           s=250, edgecolors='k', zorder=6, label='Новая точка')

# Соединения с k ближайшими соседями
for i in neighbors_idx:
    ax.plot([x_new[0], X_train[i, 0]], [x_new[1], X_train[i, 1]],
            'k--', alpha=0.7, linewidth=1.2)

# Подписи точек
labels = ['A (кл.0)', 'B (кл.0)', 'C (кл.1)', 'D (кл.1)']
for i, lab in enumerate(labels):
    ax.text(X_train[i, 0] + 0.15, X_train[i, 1] + 0.15, lab, fontsize=10)
ax.text(x_new[0] + 0.15, x_new[1] + 0.15, f'new → кл.{pred_class}',
        fontsize=10, color='green', fontweight='bold')

ax.set_xlim(0, 7); ax.set_ylim(1, 6)
ax.set_xlabel('Признак x₁')
ax.set_ylabel('Признак x₂')
ax.set_title(f'k‑NN классификация, k = {k}')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


#### 1.3.2 Регрессия

Возьмём те же четыре точки, но теперь снабдим их вещественными значениями целевой переменной:

| Точка | $x_1$ | $x_2$ | $y$  |
|-------|-------|-------|------|
| A     | 1     | 2     | 10.0 |
| B     | 2     | 3     | 20.0 |
| C     | 5     | 5     | 30.0 |
| D     | 6     | 4     | 40.0 |

Снова рассмотрим $x = (3, 3)$ и $k = 3$. Расстояния уже вычислены: ближайшие соседи — $B$, $A$, $C$. Их отклики: $y_B = 20.0$, $y_A = 10.0$, $y_C = 30.0$. Регрессионная оценка (1.4) даёт

$$
\hat{y}(x) = \frac{20.0 + 10.0 + 30.0}{3} = 20.0.
$$

Изменим $k$, чтобы увидеть влияние числа соседей. При $k=2$ соседи — $B$ и $A$, предсказание составило бы $(20.0+10.0)/2 = 15.0$. При $k=4$ в окрестность попадают все точки, и прогноз становится $(10.0+20.0+30.0+40.0)/4 = 25.0$. Этот простой пример наглядно демонстрирует, как выбор $k$ управляет степенью локальности: маленькое $k$ делает предсказание чувствительным к индивидуальным наблюдениям (низкое смещение, высокая дисперсия), тогда как большое $k$ усредняет по широкой окрестности и приближает прогноз к глобальному среднему (высокое смещение, низкая дисперсия).

Оба примера показывают, что классификация и регрессия методом k‑NN опираются на один и тот же фундаментальный шаг — выделение $k$ ближайших соседей — и различаются лишь способом агрегирования их ответов. Граница между классами в классификации будет кусочно-постоянной, а регрессионная поверхность — кусочно-постоянной (или кусочно-взвешенной при использовании весовых схем).

### 1.4 Неявные допущения метода

Прежде чем переходить к анализу гиперпараметров и алгоритмической реализации, полезно эксплицировать минимальный набор предположений, которые метод k‑NN делает относительно порождающего процесса данных.

1. **Локальная гладкость целевой функции.** Истинная функция регрессии $m(x) = \mathbb{E}[Y \mid X=x]$ (или байесовская граница в случае классификации) изменяется достаточно плавно, так что усреднение по $k$ ближайшим точкам даёт приемлемую аппроксимацию. Формально, требуется непрерывность $m(x)$ почти всюду по распределению $X$.

2. **Осмысленность метрики.** Используемая мера расстояния действительно отражает семантическую близость объектов в контексте решаемой задачи. Для признаков различной природы это, как правило, влечёт необходимость стандартизации или применения специализированных мер сходства (например, расстояние Говера для смешанных типов).

3. **Независимость и одинаковая распределённость** наблюдений $(X_i, Y_i)$ — стандартное предположение обучения с учителем, гарантирующее, что эмпирическое распределение приближает истинное.

4. **Достаточная локальная плотность данных.** В любой точке $x$, для которой мы хотим получить надёжный прогноз, объём обучающей выборки в окрестности должен быть таким, чтобы оценка $k_c/k$ (или среднее по соседям) была статистически устойчивой. Это требование напрямую ведёт к проблеме «проклятия размерности», обсуждаемой в разделе 3.

Нарушение любого из этих допущений может привести к резкому падению точности. Поэтому практическое применение метода k‑NN всегда должно начинаться с анализа того, насколько выполняются перечисленные условия, и с предобработки данных (стандартизация, отбор признаков, снижение размерности), направленной на их обеспечение.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Обучающая выборка (те же точки) и числовые отклики
X_train = np.array([
    [1, 2],   # A
    [2, 3],   # B
    [5, 5],   # C
    [6, 4]    # D
])
y_train = np.array([10.0, 20.0, 30.0, 40.0])

# Новая точка
x_new = np.array([3, 3])

# Параметр k
k = 3

# Евклидовы расстояния и поиск k ближайших соседей
distances = np.linalg.norm(X_train - x_new, axis=1)
idx_sorted = np.argsort(distances)
neighbors_idx = idx_sorted[:k]

# Предсказание — среднее арифметическое
pred_value = np.mean(y_train[neighbors_idx])

# Вывод в консоль
print("Расстояния до точек A,B,C,D:", distances)
print("Индексы k ближайших соседей:   ", neighbors_idx)
print("Отклики выбранных соседей:      ", y_train[neighbors_idx])
print("Предсказанное значение (среднее):", pred_value)

# Визуализация
fig, ax = plt.subplots(figsize=(7, 5))

# Точки, цвет соответствует значению y
sc = ax.scatter(X_train[:, 0], X_train[:, 1], c=y_train,
                cmap='viridis', edgecolors='k', s=120, zorder=5)
plt.colorbar(sc, ax=ax, label='Целевая переменная y')

# Новая точка
ax.scatter(x_new[0], x_new[1], c='red', marker='*',
           s=250, edgecolors='k', zorder=6, label='Новая точка')

# Соединения с k ближайшими соседями
for i in neighbors_idx:
    ax.plot([x_new[0], X_train[i, 0]], [x_new[1], X_train[i, 1]],
            'k--', alpha=0.7, linewidth=1.2)

# Подписи точек
labels = ['A (y=10)', 'B (y=20)', 'C (y=30)', 'D (y=40)']
for i, lab in enumerate(labels):
    ax.text(X_train[i, 0] + 0.15, X_train[i, 1] + 0.15, lab, fontsize=10)
ax.text(x_new[0] + 0.15, x_new[1] + 0.15,
        f'new → {pred_value:.1f}', fontsize=10, color='red', fontweight='bold')

ax.set_xlim(0, 7); ax.set_ylim(1, 6)
ax.set_xlabel('Признак x₁')
ax.set_ylabel('Признак x₂')
ax.set_title(f'k‑NN регрессия, k = {k}')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()



## 2. Выбор гиперпараметров

Главным гиперпараметром является **число соседей $k$**. При $k = 1$ граница решений проходит максимально близко к точкам обучающей выборки — модель имеет очень низкое смещение, но крайне высокую дисперсию: малейшее изменение состава обучения приводит к резкому изменению границы (переобучение). В противоположном крайнем случае $k = n$ предсказание всегда равно глобальному среднему (или наиболее частому классу) — смещение максимально, дисперсия нулевая (недообучение).

Формально связь с bias‑variance tradeoff можно проиллюстрировать для регрессии. Предположим, что в достаточно малой окрестности точки $x$ истинная функция отклика меняется слабо, а ошибки наблюдений некоррелированы с дисперсией $\sigma^2$. Тогда предсказание как среднее по $k$ соседям имеет дисперсию примерно $\sigma^2/k$ (при независимых ошибках), то есть с ростом $k$ дисперсия убывает. Однако при этом в окрестность попадают всё более удалённые точки, и локальное среднее перестаёт правильно отражать условное среднее в точке $x$ — растёт смещение. Оптимальное $k$ находят компромиссом, чаще всего с помощью кросс‑валидации.

Второй гиперпараметр — **метрика расстояния**. Помимо перечисленных выше, возможно использование метрики Минковского с параметром $p$; частные случаи $p=1$ (L1), $p=2$ (L2), $p\to\infty$ (равномерная норма). Важно помнить, что с ростом размерности пространства все точки становятся почти равноудалёнными (концентрация меры), поэтому различие между метриками размывается. Для текстов, представленных в виде мешка слов, часто используют косинусное расстояние. Для гетерогенных признаков можно применять взвешенные расстояния, например расстояние Говера.

Третий важный инструмент — **взвешенные соседи**. Вместо простого усреднения можно назначать веса, обратно пропорциональные расстоянию до точки:
$$
\hat{y}(x) = \frac{\sum_{i \in N_k(x)} w_i y_i}{\sum_{i \in N_k(x)} w_i}, \quad w_i = \frac{1}{d(x, x_i) + \varepsilon},
$$
где $\varepsilon$ — малая константа для предотвращения деления на ноль. Взвешивание делает предсказание более гладким и уменьшает влияние далёких соседей внутри $k$-окрестности, что особенно полезно при неравномерной плотности обучающих данных.




## 3. Проклятие размерности

Интуитивно проклятие размерности проявляется в том, что с увеличением размерности $D$ объём пространства растёт экспоненциально. Точки, равномерно распределённые в единичном гиперкубе, оказываются в среднем всё дальше друг от друга. Например, среднее расстояние между двумя случайными точками в $D$-мерном единичном гиперкубе при евклидовой метрике асимптотически равно $\sqrt{D/6}$ при $D \to \infty$. Чтобы сохранить фиксированную плотность покрытия, требуется экспоненциально больше данных.

Для метода k‑NN это означает, что понятие «близости» размывается: окрестность, содержащая $k$ ближайших соседей, становится огромной по сравнению с масштабом изменения целевой переменной. Локальная аппроксимация теряет смысл, и качество модели падает. На практике k‑NN обычно хорошо работает при $D \lesssim 20$, но для более высоких размерностей требуется либо предварительное снижение размерности (PCA), либо использование альтернативных методов.

**Углублённый взгляд: асимптотическая состоятельность k‑NN при $n \to \infty$.**  
Классический результат (Stone, 1977) утверждает, что если $k \to \infty$ и $k/n \to 0$ при $n \to \infty$, то оценка регрессии методом k‑NN является состоятельной оценкой условного среднего $\mathbb{E}[Y \mid X=x]$ для почти всех $x$ относительно распределения $X$. Аналогичное утверждение верно для классификации: риск k‑NN сходится к байесовскому риску. Однако скорость сходимости катастрофически зависит от размерности: для гладких функций регрессии она составляет $O(n^{-2/(2+D)})$. Это делает метод крайне требовательным к объёму данных при больших $D$.





## 4. Алгоритмические аспекты

Наивная реализация поиска ближайших соседей для каждого запроса перебирает все $n$ точек и вычисляет расстояния, что имеет сложность $O(D n)$ на один запрос и $O(D n^2)$ для предсказания на всём тестовом множестве — неприемлемо для больших выборок. Чтобы ускорить поиск, используют специальные пространственные структуры данных.

**KD‑дерево** (k‑dimensional tree) рекурсивно разбивает пространство гиперплоскостями, параллельными координатным осям. В каждом узле выбирается одна из размерностей и точка разбиения (обычно медиана значений признака), после чего точки разделяются на левое и правое поддеревья. При поиске $k$ ближайших соседей для запроса мы обходим дерево, спускаясь к листу, а затем поднимаемся, проверяя, может ли другой подузел содержать более близкие точки. Если расстояние от запроса до границы узла превышает текущий радиус поиска, весь узел можно отбросить без вычисления расстояний до всех его точек. Средняя сложность поиска — $O(\log n)$ для низких размерностей, но при $D$, сравнимых с $\log n$, KD‑дерево вырождается до почти полного перебора.

**Ball‑дерево** использует иерархию вложенных гиперсфер. Каждый узел хранит центр и радиус минимальной сферы, покрывающей все точки поддерева. При поиске отсечение происходит, если расстояние от запроса до сферы узла больше текущего радиуса. Ball‑деревья обычно работают лучше KD‑деревьев в высоких размерностях.

Псевдокод алгоритма поиска ближайшего соседа в KD‑дереве выглядит так:

```
def search(node, query, best):
    если node — лист:
        обновить best расстоянием до точки
        вернуть
    определить, в какой дочерний узел спускаться первым
    search(ближайший_дочерний, query, best)
    если расстояние до разделяющей границы другого дочернего < best:
        search(другой_дочерний, query, best)
```





## 5. Реализация с нуля и сравнение с библиотечной версией

Приведём реализацию k‑NN классификатора с нуля на Python. Будем использовать евклидово расстояние, единичные или взвешенные голоса.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

class SimpleKNN:
    def __init__(self, n_neighbors=5, weights='uniform'):
        self.n_neighbors = n_neighbors
        self.weights = weights

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        # матрица расстояний (n_query x n_train)
        dist = np.linalg.norm(X[:, np.newaxis, :] - self.X_train[np.newaxis, :, :], axis=2)
        # индексы k ближайших
        idx = np.argsort(dist, axis=1)[:, :self.n_neighbors]
        k_labels = self.y_train[idx]
        if self.weights == 'uniform':
            # большинство голосов
            pred = np.apply_along_axis(lambda x: np.bincount(x).argmax(), axis=1, arr=k_labels)
        else:
            # взвешенные веса
            dist_k = np.take_along_axis(dist, idx, axis=1)
            weights = 1.0 / (dist_k + 1e-8)
            # для каждого класса суммируем веса
            classes = np.unique(self.y_train)
            weighted_votes = np.zeros((X.shape[0], len(classes)))
            for j, c in enumerate(classes):
                mask = (k_labels == c)
                weighted_votes[:, j] = np.sum(weights * mask, axis=1)
            pred = classes[np.argmax(weighted_votes, axis=1)]
        return pred
```

Проверим на реальном датасете breast_cancer и сравним с реализацией sklearn.

```python
# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Стандартизация
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Наша реализация
my_knn = SimpleKNN(n_neighbors=5)
my_knn.fit(X_train_sc, y_train)
my_pred = my_knn.predict(X_test_sc)
print("My KNN accuracy:", accuracy_score(y_test, my_pred))

# Sklearn
sk_knn = KNeighborsClassifier(n_neighbors=5)
sk_knn.fit(X_train_sc, y_train)
sk_pred = sk_knn.predict(X_test_sc)
print("Sklearn KNN accuracy:", accuracy_score(y_test, sk_pred))
```

Результаты должны совпадать (с точностью до возможного различия в весовых схемах). Теперь визуализируем границы решений на синтетических двумерных данных (moons) для разных $k$.

```python
X_moons, y_moons = make_moons(n_samples=200, noise=0.2, random_state=42)
X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(X_moons, y_moons, test_size=0.3, random_state=42)

def plot_decision_boundary(model, X, y, ax, title):
    h = 0.05
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='viridis')
    ax.set_title(title)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for k, ax in zip([1, 5, 15], axes):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_m, y_train_m)
    plot_decision_boundary(knn, X_train_m, y_train_m, ax, f'k = {k}')
plt.tight_layout()
plt.show()
```

Графики покажут, что при $k=1$ граница сильно изрезана и присутствуют «островки» (высокая дисперсия), при $k=15$ граница становится более гладкой, но хуже подстраивается под структуру данных (смещение).

---

## 6. Эксперименты: влияние $k$, метрики и размерности

Продолжим исследование на breast_cancer. Построим кривую зависимости точности на обучении и тесте от $k$.

```python
train_acc, test_acc = [], []
ks = range(1, 31)
for k in ks:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_sc, y_train)
    train_acc.append(accuracy_score(y_train, knn.predict(X_train_sc)))
    test_acc.append(accuracy_score(y_test, knn.predict(X_test_sc)))

plt.figure(figsize=(8,5))
plt.plot(ks, train_acc, label='Train')
plt.plot(ks, test_acc, label='Test')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.title('Влияние k на точность')
plt.show()
```

Типичная картина: на обучении точность максимальна при $k=1$ и снижается с ростом $k$; на тесте наблюдается пик в районе $k \approx 5-10$. Это наглядно демонстрирует компромисс смещения и дисперсии.

Сравним разные метрики:

```python
metrics = ['euclidean', 'manhattan', 'cosine']
for m in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=m)
    knn.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_sc))
    print(f"Metric {m}: accuracy = {acc:.4f}")
```

Для низкоразмерных данных с числовыми признаками евклидова и манхэттенская метрики дают схожие результаты; косинусная может работать хуже, если не применяется специальная нормировка.

Важность стандартизации легко увидеть, сравнив точность на нестандартизованных и стандартизованных данных:

```python
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)               # без стандартизации
acc_raw = accuracy_score(y_test, knn.predict(X_test))
knn.fit(X_train_sc, y_train)
acc_sc = accuracy_score(y_test, knn.predict(X_test_sc))
print(f"Без стандартизации: {acc_raw:.4f}, со стандартизацией: {acc_sc:.4f}")
```

Без стандартизации точность значительно падает, так как признаки с большим масштабом подавляют вклад остальных в расстояние.

Для демонстрации проклятия размерности сгенерируем синтетические данные с двумя гауссовскими классами в пространствах разной размерности:

```python
from sklearn.datasets import make_classification

dims = [2, 10, 50, 100, 200]
accs = []
for D in dims:
    X_s, y_s = make_classification(n_samples=500, n_features=D, n_informative=D, n_redundant=0,
                                   n_clusters_per_class=1, random_state=42)
    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X_s, y_s, test_size=0.3, random_state=42)
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train_s)
    X_test_s = scaler.transform(X_test_s)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train_s, y_train_s)
    acc = accuracy_score(y_test_s, knn.predict(X_test_s))
    accs.append(acc)
    print(f"D={D}: accuracy={acc:.3f}")

plt.figure(figsize=(8,5))
plt.plot(dims, accs, 'o-')
plt.xlabel('Число признаков D')
plt.ylabel('Accuracy')
plt.title('Проклятие размерности для k‑NN')
plt.grid(alpha=0.3)
plt.show()
```

С ростом $D$ точность k‑NN монотонно падает: при $D=200$ она часто приближается к случайному угадыванию.

---

## 7. Связь с байесовским подходом и другими методами

Метод k‑NN можно вывести как простейшую непараметрическую оценку плотности. Пусть в окрестности объёма $V$ точки $x$ находится $k$ объектов. Тогда оценка плотности в точке $x$ равна $\hat{p}(x) = \frac{k}{n V}$. Отношение числа соседей разных классов даёт оценку отношения условных плотностей, а решение по большинству голосов аппроксимирует байесовское решающее правило.

**Углублённый взгляд: сходимость к байесовскому классификатору.**  
При $k \to \infty$ и $k/n \to 0$ (при $n \to \infty$) оценка плотности сходится к истинной в каждой точке, следовательно, и классификатор k‑NN стремится к оптимальному байесовскому классификатору. Строгое доказательство основано на законе больших чисел и свойствах локального полиномиального сглаживания. Этот результат важен теоретически, но на практике ограничен проклятием размерности.

По сравнению с логистической регрессией и SVM метод k‑NN является полностью непараметрическим: он не строит явной глобальной модели и не оценивает параметры. Это делает его чрезвычайно гибким, но одновременно лишает интерпретируемости: нельзя сказать, какой признак важен, а граница решений существует лишь в виде набора точек. При этом k‑NN часто служит отличным «бенчмарком»: если более сложная модель не превосходит k‑NN, её применение вряд ли оправдано.

---

## Вопросы для самопроверки

### Для начинающих
1. Как работает k‑NN для классификации и регрессии?
2. Что происходит с границей решений при $k=1$ и при $k=n$?
3. Какие метрики расстояния вы знаете и в каких ситуациях они применяются?
4. Почему перед применением k‑NN необходимо стандартизировать признаки?
5. В чём проявляется проклятие размерности для k‑NN?
6. Как выбрать число соседей $k$ на практике?

### Для профессионалов
1. Докажите, что k‑NN является состоятельной оценкой условного среднего при $k\to\infty$ и $k/n\to0$.
2. Почему KD‑дерево деградирует при большой размерности? Дайте формальное обоснование.
3. Выведите оптимальную весовую схему для регрессии, минимизирующую локальную среднеквадратичную ошибку.
4. Сравните k‑NN с ядерной оценкой Надарая–Ватсона: в чём сходство и различие?
5. Как k‑NN можно интерпретировать как оценку отношения плотностей для классификации?

## Задания повышенной сложности
1. Реализуйте KD‑дерево для поиска k ближайших соседей и сравните время работы с наивным перебором при различных $n$ и $D$.
2. Докажите, что среднее евклидово расстояние между двумя равномерно распределёнными точками в $D$-мерном единичном гиперкубе асимптотически равно $\sqrt{D/6}$.
3. Постройте кривые обучения для k‑NN и логистической регрессии на breast_cancer; проанализируйте, как размер выборки влияет на смещение и дисперсию каждого метода.
4. Реализуйте взвешенную k‑NN регрессию и сравните её с обычной на данных с неравномерной плотностью признаков.
5. Докажите, что в пределе $n\to\infty, k\to\infty, k/n\to0$ байесовская ошибка классификации для k‑NN достигается (риск сходится к байесовскому риску).